# Day 7 / 42: Data Cleaning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VaishnaviJagtap18/42-days-aiml-challenge/blob/main/week1_data_foundations/day07_data_cleaning/day07_notebook.ipynb)

**#42DaysOfML**

---

## What You Will Learn
- Run a full data audit before touching any cleaning logic
- Find and remove exact duplicates
- Standardize inconsistent string formatting
- Use fuzzy matching for near-duplicate text values
- Detect contradictory records (age vs birth year)
- Validate your cleaned dataset with assertions

**Dataset:** A deliberately dirty customer dataset with all 5 real-world cleaning issues baked in.

---


## Step 0: Install and Import

In [ ]:
# Install fuzzywuzzy for near-duplicate detection
!pip install fuzzywuzzy python-Levenshtein -q

import pandas as pd
import numpy as np
from fuzzywuzzy import fuzz, process
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("All libraries loaded successfully.")


---
## Step 1: Load the Dirty Dataset and Run the Audit

**Rule 1:** Before you clean anything, measure everything.
Print shape, dtypes, nulls, duplicate count, and value distributions for key columns.
This audit tells you where to spend your time.


In [ ]:
# Load the dirty dataset
# Download from GitHub repo or create it inline below

import io, requests

url = "https://raw.githubusercontent.com/VaishnaviJagtap18/42-days-aiml-challenge/main/week1_data_foundations/day07_data_cleaning/dirty_customer_data.csv"

try:
    df = pd.read_csv(url)
    print("Loaded from GitHub.")
except:
    # Fallback: generate inline if URL not available yet
    np.random.seed(42)
    n = 200
    cities = ["Mumbai", "mumbai", "MUMBAI", "Mumbai ", " mumbai",
              "Pune", "pune", "PUNE", "Delhi", "delhi",
              "New Delhi", "Bangalore", "Bengaluru", "bangalore"]
    data = {
        "customer_id": list(range(1, n+1)),
        "name": ["Customer_" + str(i) for i in range(1, n+1)],
        "age": [float(np.random.randint(18, 65)) for _ in range(n)],
        "birth_year": [2024 - np.random.randint(18, 65) for _ in range(n)],
        "salary": [float(np.random.randint(30000, 200000)) for _ in range(n)],
        "city": [np.random.choice(cities) for _ in range(n)],
        "email": ["user" + str(i) + ("@gmail.com" if i % 7 != 0 else "@GMAIL.COM") for i in range(1, n+1)],
        "phone": [str(np.random.randint(7000000000, 9999999999)) if i % 5 != 0 else "NA" for i in range(1, n+1)],
        "purchase_amount": [round(np.random.uniform(100, 50000), 2) for _ in range(n)],
        "join_date": ["2023-0" + str(np.random.randint(1,9)) + "-" + str(np.random.randint(10,28)) for _ in range(n)]
    }
    df = pd.DataFrame(data)
    dup_rows = df.iloc[5:10].copy()
    df = pd.concat([df, dup_rows], ignore_index=True)
    df.loc[10:14, "birth_year"] = 1960
    df.loc[15:19, "salary"] = np.nan
    df.loc[20:22, "age"] = np.nan
    df.loc[30:34, "email"] = df.loc[30:34, "email"].apply(lambda x: "  " + x + "  ")
    print("Generated inline fallback dataset.")

print("\nShape:", df.shape)
print("\nData types:")
print(df.dtypes)


In [ ]:
# Null count per column
print("Null counts:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())


In [ ]:
# Value distribution for the 'city' column — classic dirty text column
print("City column value counts (before cleaning):")
print(df["city"].value_counts())

# This is the problem: Mumbai, mumbai, MUMBAI, 'Mumbai ' are all the same city
# Your model sees them as 7 different cities


---
## Step 2: Remove Exact Duplicates

Exact duplicates inflate your training data. A model trained on 5 copies of the same row
learns that row 5 times harder than the rest. That skews your loss function.


In [ ]:
rows_before = len(df)

df = df.drop_duplicates()

rows_after = len(df)
print(f"Removed {rows_before - rows_after} exact duplicate rows.")
print(f"Shape after: {df.shape}")


---
## Step 3: Standardize String Formatting

Strip whitespace, normalize case, fix dtype issues.
One inconsistent value in a categorical column creates a new category your model has never seen
in production.


In [ ]:
# Fix city column: strip whitespace + title case
df["city"] = df["city"].str.strip().str.title()

# Fix email: strip whitespace + lowercase
df["email"] = df["email"].str.strip().str.lower()

# phone column has "NA" strings — convert to actual NaN
df["phone"] = df["phone"].replace("NA", np.nan)

print("City after formatting:")
print(df["city"].value_counts())

print("\nEmail sample (first 5):")
print(df["email"].head())


---
## Step 4: Near-Duplicate Detection with Fuzzy Matching

After basic formatting you still have: `Delhi` and `New Delhi`, `Bangalore` and `Bengaluru`.
These refer to the same or overlapping geographic areas depending on your use case.

Fuzzy matching compares string similarity using edit distance.
A score of 75+ means the strings are likely the same thing spelled differently.


In [ ]:
# Get unique city names after basic formatting
unique_cities = df["city"].unique().tolist()
print("Unique cities after formatting:", sorted(unique_cities))

# Build a canonical mapping using fuzzy matching
city_map = {}
canonical_list = []

for city in sorted(unique_cities):
    if not canonical_list:
        canonical_list.append(city)
        city_map[city] = city
    else:
        match, score = process.extractOne(city, canonical_list, scorer=fuzz.ratio)
        print(f"  '{city}' vs '{match}': score={score}")
        if score >= 80:  # threshold: adjust based on your domain knowledge
            city_map[city] = match
        else:
            canonical_list.append(city)
            city_map[city] = city

print("\nFinal city mapping:")
for k, v in city_map.items():
    print(f"  '{k}' -> '{v}'")


In [ ]:
# Apply the mapping
df["city"] = df["city"].map(city_map)

print("City column after fuzzy standardization:")
print(df["city"].value_counts())

# NOTE: 'New Delhi' and 'Delhi' have a score below 80 (they are different strings).
# Whether to merge them depends on your domain: for a city-level model they may be
# the same. For a pincode-level model they are different. THIS is a domain decision,
# not a data decision. Always check with a domain expert before merging.


---
## Step 5: Detect Contradictory Records

Age=25 but birth_year=1960 means the row is internally inconsistent.
A model that trains on this learns a false signal. Fix it or flag it.


In [ ]:
current_year = 2024
df["age_from_birth"] = current_year - df["birth_year"].astype(int)

# Flag rows where stated age differs from birth year by more than 2 years
df["age_mismatch"] = abs(df["age"].fillna(df["age_from_birth"]) - df["age_from_birth"]) > 2

print("Rows with age-birth_year mismatch:", df["age_mismatch"].sum())
print("\nSample contradictory rows:")
print(df[df["age_mismatch"]][["customer_id", "age", "birth_year", "age_from_birth"]].head(10))


In [ ]:
# Fix: trust birth_year as ground truth, correct the age column
df.loc[df["age_mismatch"], "age"] = df.loc[df["age_mismatch"], "age_from_birth"]

# Verify fix
remaining_mismatches = abs(df["age"] - df["age_from_birth"]) > 2
print("Mismatches after fix:", remaining_mismatches.sum())

# Drop helper columns
df = df.drop(columns=["age_from_birth", "age_mismatch"])


---
## Step 6: Validate with Assertions

Assertions are your automated quality gate. Run them at the end of every cleaning pipeline.
If any assertion fails, you catch the bug before the model does.


In [ ]:
# Final validation
try:
    # No duplicates
    assert df.duplicated().sum() == 0, "Duplicates still exist!"
    
    # Email has no whitespace
    assert df["email"].str.contains("  ").sum() == 0, "Email whitespace found!"
    
    # Email is lowercase
    assert (df["email"] == df["email"].str.lower()).all(), "Emails not lowercase!"
    
    # Age is within valid range (18-65 for this dataset)
    assert df["age"].dropna().between(17, 66).all(), "Age out of expected range!"
    
    # City column has only standardized values
    assert df["city"].nunique() <= 5, f"Too many city values: {df['city'].unique()}"
    
    print("All assertions passed.")
    print(f"\nFinal clean dataset shape: {df.shape}")
    print("\nFinal null counts:")
    print(df.isnull().sum())
    
except AssertionError as e:
    print(f"Assertion failed: {e}")


---
## Step 7: Before vs After Comparison


In [ ]:
# Visual comparison of city column before and after
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Reload original to show before state
import io
df_original = pd.read_csv(io.StringIO(open.__doc__ or "")) if False else None

# Show after
city_counts = df["city"].value_counts()
axes[0].barh(city_counts.index, city_counts.values, color="#00C4CC")
axes[0].set_title("City Column AFTER Cleaning", fontsize=13, fontweight='bold')
axes[0].set_xlabel("Count")

# Null counts after
null_counts = df.isnull().sum()
null_counts = null_counts[null_counts > 0]
if len(null_counts) > 0:
    axes[1].bar(null_counts.index, null_counts.values, color="#F4A233")
    axes[1].set_title("Remaining Nulls After Cleaning", fontsize=13, fontweight='bold')
    axes[1].set_ylabel("Count")
    axes[1].tick_params(axis='x', rotation=15)
else:
    axes[1].text(0.5, 0.5, "No remaining nulls
(except phone - valid missing)",
                ha='center', va='center', fontsize=12, color='green',
                transform=axes[1].transAxes)
    axes[1].set_title("Remaining Nulls After Cleaning", fontsize=13, fontweight='bold')
    axes[1].axis('off')

plt.tight_layout()
plt.savefig("day07_cleaning_result.png", dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved.")


---
## Practice Exercise

Take the cleaned dataset and:

1. Check if `join_date` is stored as a string (it is). Convert it to proper `datetime` dtype using `pd.to_datetime()`.
2. Extract `join_month` and `join_year` as separate integer columns.
3. Verify: are there any `join_date` values in the future (after today)?
4. Write one assertion to confirm `join_year` is always between 2020 and 2024.

**Post your solution in the comments on LinkedIn with your observations.**

---


---
## Production Rule Summary

| Issue | Detection | Fix |
|---|---|---|
| Exact duplicates | `df.duplicated().sum()` | `drop_duplicates()` |
| Inconsistent case | `value_counts()` | `.str.strip().str.title()` |
| Near-duplicates | fuzzy matching | canonical map |
| Wrong dtypes | `df.dtypes` | explicit cast |
| Contradictory records | column cross-check | trust ground truth column |
| Validation | assertions | `assert` at pipeline end |

**Key rule for production:** Every cleaning step must be reproducible. Write it as a function,
not a one-time script. New data arrives every day. Your pipeline runs on it automatically.

---

See you on **Day 8: Week 1 Revision** — 5 MNC interview questions, 3 practice projects combining
everything from this week, and the community challenge.

**GitHub repo:** https://https://github.com/VaishnaviJagtap18/42-Days-0f-ML-Challenge

#42DaysOfML #MachineLearning #DataScience #Python #MLEngineer
